## Solving the OneMax problem using a GA 

### Constants and Imports
We will need various constants such as the population size, crossover rate, mutation rate etc. along with imports for the packages we are going to make use of.
Typically these are passed as parameters to the algorithms but we're keeptrying to keep things simple for the time being.

In [1]:
import string
import random
from tokenize import String

from sympy.parsing.sympy_parser import null
from torch.linalg import solve

random.seed(42) # initialise and make repeatable for testing purposes
from operator import itemgetter # may be useful in retrieving elements of the population

#Population size and other constants - these can be changed.
pop_size=50
crossover_rate = 0.75
mutation_rate = 0.05

### Representation
Always the first thing to think about - how do we represent a solution to the problem - i.e. how should this be encoded and what data type (or types) should be used. It can often help to think about how you might manipulate this (to generate new solutions) and evaluate it. <br>
However this case is easy as the problem and the representation are identical - a string of 1's and 0's - so nothing else needs to be done here.

### Fitness Function
The fitness function defines how we score or evaluate a solution and has an important role is guiding the search towards the solution. In particular it is essential that we are able to establish how much better one solution is than another.<br>

You will already have defined a fitness function in the previous exercise and can just reuse it here.

In [462]:
# Function to calculate a candidates solution fitness - solutions closer to the goal should score higher
# Input parameters: candidate solution (string of 1's and 0's)
# Returns: fitness of the candidate solution
def fitness(solution):
    # Your code goes here
    score = 0
    for i in solution:
        if i == "1":
            score += 1
    score = score / len(solution) * 100
    return score

# test the above (precise results will depend on your function and may vary with string length)
print(fitness("1111111111")) # output should be the highest score
print(fitness("1010110010")) # output should be a mid-range score
print(fitness("0000000000")) # output should be the lowest score
print(fitness("11111")) # output should be the highest score
print(fitness("10101")) # output should be a mid-range score
print(fitness("00000")) # output should be the lowest score
print(fitness("1")) # output should be the highest score
print(fitness("0")) # output should be the lowest score

100.0
50.0
0.0
100.0
60.0
0.0
100.0
0.0


## The GA algorithm
The main steps are:

- Generate a random population
- Evaluate the fitness of the population
- While a solution has not been found
    - Select and crossover individuals to create a new population
    - Mutate some individuals of the new population
    - Make the new population the current population
    - Evaluate the fitness of the current population
- Return the best individual and its fitness

We will develop functions to help us carry out these individual steps below before calling them in the main loop (this is defined at the end of this notebook)

## Generate a random population
The initial step is to generate a population of random solutions. We are going to store the population as a list of tuples so that we can associate a fitness value with each element (rather than calling the fitness function multiple times). 

We first need a function to generate a random binary string before defining the function to use this to create the initial population.

These two steps are defined below:

### Generate an Individual
Create a random string of 1's and 0's of length defined in the parameter.
Again you should already have a function which does this.

In [162]:
# Generate a random string of 1's and 0's of length defined by the parameter
# Input parameters: length of the string to generate
# Returns: The randomly generated string
def gen_individual(length):
    # Your code goes here
    solution = ""
    for i in range(length):
        solution += str(random.randint(0,1))
    return solution

# test... run this a few times to check the output is as expected
print(gen_individual(1))
print(gen_individual(5))
print(gen_individual(10))

0
11010
0100011010


### Create the population
Using the gen_individual() function above, build a population from individuals which are represented as lists [string, fitness]. <br>
The fitness will initially be 0 as we have not calculated it yet. The population itself will be a list of these lists.

i.e. population = [["individual1", fitness1], ["individual2", fitness2], ["individual3", fitness3], ... etc.] 

The size of the population is determined by the pop-size constant defined at the start of this notebook.<br>
The length of the individual to be generated is passed in as a parameter.

In [223]:
# Input parameters: The length of the individuals (strings) to generate. The population size is determined by the global variable pop_size.
# Returns: A population of randomly generated individuals with associated fitness values (set to 0)
def gen_population(string_length):
    # Your code goes here
    population = []
    for i in range(pop_size):
        population.append([gen_individual(string_length) , 0])
    return population
# test
print(gen_population(10))
# print(gen_population(5))
# print(gen_population(10))

[['1110001011', 0], ['0000100011', 0], ['0011010100', 0], ['0000010010', 0], ['1011110001', 0], ['0111001001', 0], ['1001110010', 0], ['1011001100', 0], ['0000100011', 0], ['0111010000', 0], ['1101010111', 0], ['0110110100', 0], ['1010000011', 0], ['0011011011', 0], ['0000011101', 0], ['0110000100', 0], ['1110110011', 0], ['1101010011', 0], ['1100011100', 0], ['1111011001', 0], ['0101010010', 0], ['1011111000', 0], ['0101111011', 0], ['1001100011', 0], ['0001101010', 0], ['1101100100', 0], ['1100011101', 0], ['1110111100', 0], ['0100011000', 0], ['1010010110', 0], ['1110110000', 0], ['1011001110', 0], ['0100100101', 0], ['1010010110', 0], ['0110011101', 0], ['1111100111', 0], ['1111110010', 0], ['0110000001', 0], ['1000000110', 0], ['0011100100', 0], ['1111110101', 0], ['1111000101', 0], ['1000000000', 0], ['0001001000', 0], ['1110011001', 0], ['1010011111', 0], ['1110100100', 0], ['1100100101', 0], ['0010010001', 0], ['0000100110', 0]]


### Evaluate the population
Go through each element in the population, calculating its fitness and associating the fitness value with the individual in the tuple.<br>
i.e. the in the illustration of the population above, fitness1 will be the fitness value of "individual1" and so on.

In [311]:
# Input parameters: A population of individuals (solutions)
# Returns: The population modified with the computed fitness of each individual
def evaluate_pop(pop):
    # Your code goes here
    newpopu=[]
    for item in pop:
        item[1] = fitness(item[0])
        newpopu.append(item)
    return newpopu


# Test
newpop = gen_population(10)
print(evaluate_pop(newpop))
# newpop = gen_population(5)
# print(evaluate_pop(newpop))
# newpop = gen_population(10)
# print(evaluate_pop(newpop))

[['1101011010', 60.0], ['1110110000', 50.0], ['1001101110', 60.0], ['0011010101', 50.0], ['0111110110', 70.0], ['1111111010', 80.0], ['1101000111', 60.0], ['0101011110', 60.0], ['1101001000', 40.0], ['0000010100', 20.0], ['0000011110', 40.0], ['1110100000', 40.0], ['0110110010', 50.0], ['0111101101', 70.0], ['1011001011', 60.0], ['1110010100', 50.0], ['0010100111', 50.0], ['0110000110', 40.0], ['0110010101', 50.0], ['1101101111', 80.0], ['0101010100', 40.0], ['0000001011', 30.0], ['0111100000', 40.0], ['0000100100', 20.0], ['0001001101', 40.0], ['0100011000', 30.0], ['0001110100', 40.0], ['0100000110', 30.0], ['0111111001', 70.0], ['0011110011', 60.0], ['0001110011', 50.0], ['0000100011', 30.0], ['0111111101', 80.0], ['1000100100', 30.0], ['0101111100', 60.0], ['1000110101', 50.0], ['0010111010', 50.0], ['1000100011', 40.0], ['1111111010', 80.0], ['0111110101', 70.0], ['1100001000', 30.0], ['0001010110', 40.0], ['0000001100', 20.0], ['0100010101', 40.0], ['1100001000', 30.0], ['1000100

We will also need a function that returns the best individual in the population and its associated fitness.

In [597]:
# Input parameters: A population of individuals (solutions)
# Returns: The fittest individual and its score 
def fittest_individual(pop):
    # Your code goes here
    i = 0
    popFitness = 0
    bestFitness = []
    for item in pop:
        i +=1
        if fitness(item[0]) > popFitness:
            popFitness = fitness(item[0])
            bestFitness = [item[0], popFitness]
    return bestFitness



#test
newpop = evaluate_pop(gen_population(10))
# print(newpop)
print(fittest_individual(newpop))

['1110111111', 90.0]


### Selection
The selection of candidate parent solutions is going to be very simple - we are going to randomly select from the top 50% of solutions.<br>
Since the population size needs to remain constant over generations the number of parents needs to be equal to the size of the population.Consequently, there is a chance that some individuals may be chosen more than once whereas others may never be selected as parents - a strategy known as selection with replacement (i.e. we never actually remove an individual from the pool of possible parents). 

At this stage, all we need to do is  define a function to return the top 50% of the population.


In [674]:
# Orders the list according to fitness and returns the top 50%
# Input parameters: A population of individuals
# Returns: The subset of the population drawn from the top 50% of the individuals (as ranked by fitness)
def find_top_50(pop):
    # Your code goes here
    print(pop)

#test
newpop = evaluate_pop(gen_population(10))
# print(newpop)
print(find_top_50(newpop))

[['1011100000', 40.0], ['1110110001', 60.0], ['1000100100', 30.0], ['0111001000', 40.0], ['0101000001', 30.0], ['1010110000', 40.0], ['0100100110', 40.0], ['0010010100', 30.0], ['1101000100', 40.0], ['0111101011', 70.0], ['0100011101', 50.0], ['0001000100', 20.0], ['1100110100', 50.0], ['0111100101', 60.0], ['0000011111', 50.0], ['0001010001', 30.0], ['0100001111', 50.0], ['0110000010', 30.0], ['1110001110', 60.0], ['0011010000', 30.0], ['1001010100', 40.0], ['1110111000', 60.0], ['0111101001', 60.0], ['1101010100', 50.0], ['1000101011', 50.0], ['1111100011', 70.0], ['1101000001', 40.0], ['1001101110', 60.0], ['0111010111', 70.0], ['1101101101', 70.0], ['1011100000', 40.0], ['0111000101', 50.0], ['1000001011', 40.0], ['1011001111', 70.0], ['1010001110', 50.0], ['1100111110', 70.0], ['1101011011', 70.0], ['1001101011', 60.0], ['0011111010', 60.0], ['0011001101', 50.0], ['1010111001', 60.0], ['1000101111', 60.0], ['0100000111', 40.0], ['1110111001', 70.0], ['0100111000', 40.0], ['0010111

### Recombine individuals to create the next generation
This step involves two sub-steps - the identification of pairs of parents and recombination to create offspring which we will tackle in reverse order.

- The first step is to define the recombination step (the crossover operation) which is called repeatedly. 
Not all parents recombine - a feature which is governed by the crossover rate which in this case is coded as a constant (e.g. 0.75). You will need to generate a random number and compare it against the crossover rate to determine if crossover takes place. If it does, then parents are recombined to generate offspring which are returned by the function. If it doesn’t, then the unmodified parents are returned. In both cases, the returned values will form part of the new population (we don't need to keep track of what has happened).


In [ ]:
# Input parameters: Two parents (tuples of the form <individual, fitness>)
# Returns: Two children after combining the two parents (tuples of the form <individual, 0>)
# or the original two parents if the crossover threshold is not met
def crossover(parent1, parent2):
    # Your code goes here

# test 
# print(crossover(("11001100",4),("10101010",4)))


 - The second step is the select_and_generate_new_population function which repeatedly (and randomly) selects pairs of parents from the previously selected pool of parents which are recombined (by calling the crossover operation above) to form new individuals (children) which will be used to build the new population. 


In [ ]:
# Input parameters: a population
# Returns: The new population after selection and crossover (using the operation defined above) have been applied
def select_and_generate_new_population(parents):
    # Your code goes here

# test
# newpop = evaluate_pop(gen_population(10))
# print(newpop)
# print(select_and_generate_new_population(newpop)

### Mutate the new population
This takes place once the new population is completely generated and is governed by the mutation rate (typically quite small, such as 0.05). Again this is provided as a constant in our implementation. There are several mutation operations that may be applied in this case but change should be quite small. Thinking about the “neighbourhood” in the hill climber can help you decide on what form this should take.


In [ ]:
# Function to mutate individuals in the population according to the mutation rate
# Input parameters: A population
# Returns: A modified population
def mutate(pop):
    # Your code goes here

# test
# newpop = evaluate_pop(gen_population(10))
# print(newpop)
# print(select_and_generate_new_population(newpop)

### Main GA Loop

We are now in a position code the main GA algorithm which, as we said at the start, should take the following form:
- Generate a random population
- Evaluate the fitness of the population
- While a solution has not been found
    - Select and crossover individuals to create a new population
    - Mutate some individuals of the new population
    - Make the new population the current population
    - Evaluate the fitness of the current population
- Return the best individual and its fitness

In [ ]:
# Main GA loop which takes the string length as an input parameter.
# When developing these it is helpful to print out the fittest individual at each generation to see how the GA is progressing. 
# #This has been commented out in the code below but can be uncommented for debugging purposes.

def GA(string_length):
    pop = gen_population(string_length)
    scored_pop = evaluate_pop(pop)
    i = 0
    # print("Generation",i)   # Uncomment these lines to see the progress of the GA
    # print(fittest_individual(scored_pop))
    while fittest_individual(scored_pop)[1] <string_length: # This would usually be more general such a monitoring the change in fitness over iterations but keeping thngs simple here
        i += 1
        new_pop = []
        parents = find_top_50(scored_pop)
        new_pop = select_and_generate_new_population(parents)
        mutated_pop = mutate(new_pop)
        pop = mutated_pop
        scored_pop = evaluate_pop(pop)
        # print("Generation",i)  # Uncomment these lines to see the progress of the GA
        # print(fittest_individual(scored_pop))
    return fittest_individual(scored_pop)

# tests
# GA(1)
# GA(10)
# GA(25)
# GA(100)
    